# 实验报告03：正则表达式与网络结构化数据

- **学号**：20244081108
- **姓名**：王亨泰
- **环境**：独立 conda 虚拟环境 `pyda`（Python 3.11 + Jupyter），不使用 base 环境

## 1.1 第一个代码框：身份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244081108

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "pyda")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()} | {platform.machine()} | {platform.system()} | {uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID} - {MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] pyda
[学号] 20244081108
[机器指纹] 620bd7970f541ccf
[RUN_ID] 20244081108 - 620bd7970f541ccf
[Python] 3.11.17


---
## 二、实验内容与过程

## 实验1：正则表达式的基础匹配

先导入 `re` 模块。

In [2]:
import re

### 1. 数字、空白与整词匹配

In [3]:
text1 = "cat on mat; later: cats and dogs"

print(r"\d+   匹配结果:", re.findall(r"\d+", text1))
print(r"\s+   匹配结果:", re.findall(r"\s+", text1))
print(r"\bcat\b 匹配结果:", re.findall(r"\bcat\b", text1))

\d+   匹配结果: []
\s+   匹配结果: [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b 匹配结果: ['cat']


**规则说明**：

- `\d+` 匹配一个或多个连续数字；该文本没有数字，所以结果为空列表。
- `\s+` 匹配一个或多个连续空白字符，返回每一段空白。
- `\bcat\b` 用单词边界 `\b` 限定整词，只匹配独立的 `cat`；`cats` 因为后面还有字母 `s`，不满足右单词边界，所以不匹配。

### 2. 交替 `|` 与匹配顺序

In [4]:
print(r"cat|dog 在 'cat dog bird CAT' 上:", re.findall(r"cat|dog", "cat dog bird CAT"))
print(r"a|a|a|a|a 在 'aaaa' 上:", re.findall(r"a|a|a|a|a", "aaaa"))
print(r"aaa|a|a 在 'aaaa' 上:", re.findall(r"aaa|a|a", "aaaa"))

cat|dog 在 'cat dog bird CAT' 上: ['cat', 'dog']
a|a|a|a|a 在 'aaaa' 上: ['a', 'a', 'a', 'a']
aaa|a|a 在 'aaaa' 上: ['aaa', 'a']


**规则说明**：

- `cat|dog` 匹配 `cat` 或 `dog`；正则默认大小写敏感，大写的 `CAT` 不匹配。
- 交替匹配时按从左到右的顺序尝试，且每次匹配后从匹配结束位置继续。
- `a|a|a|a|a` 每个分支都只匹配一个 `a`，于是逐个匹配，得到 4 个 `a`。
- `aaa|a|a` 第一个分支 `aaa` 优先，先吃掉前 3 个 `a`，剩下 1 个再由 `a` 匹配，所以得到 `['aaa', 'a']`。这说明**交替分支的顺序会影响结果**。

### 3. 各种量词的含义

In [5]:
text3 = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]
for pat in patterns:
    print(f"{pat:10} -> {re.findall(pat, text3)}")

ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


**量词说明**：

| 量词 | 含义 | 本例匹配 |
|---|---|---|
| `ca*t` | `*`：前面的 `a` 出现 **0 次或多次** | ct、cat、caat、caaat |
| `ca+t` | `+`：`a` 出现 **1 次或多次** | cat、caat、caaat（ct 不匹配） |
| `ca?t` | `?`：`a` 出现 **0 次或 1 次** | ct、cat |
| `ca{2}t` | `{2}`：`a` 恰好出现 **2 次** | caat |
| `ca{1,2}t` | `{1,2}`：`a` 出现 **1 到 2 次** | cat、caat |

---
## 实验2：分组、回溯引用与替换

### 1. 用捕获组提取邮箱的用户名和域名

In [6]:
mail_text = "联系 : alice@example.com ; 备选 : bob@test.org"
mail_pattern = r"([A-Za-z0-9_.+-]+)@([A-Za-z0-9.-]+)"

for m in re.finditer(mail_pattern, mail_text):
    print("完整邮箱:", m.group(0))
    print("  用户名(group 1):", m.group(1))
    print("  域名  (group 2):", m.group(2))

完整邮箱: alice@example.com
  用户名(group 1): alice
  域名  (group 2): example.com
完整邮箱: bob@test.org
  用户名(group 1): bob
  域名  (group 2): test.org


**规则说明**：一对圆括号形成一个捕获组。`@` 前的第 1 组匹配用户名，`@` 后的第 2 组匹配域名；`group(0)` 是整个匹配，`group(1)`、`group(2)` 分别是两个括号捕获的内容。

### 2. 用后向引用找出连续重复的单词

In [7]:
repeat_text = "cat cat, dog cat, bird bird"
# (\w+) 捕获一个单词，\s+ 跳过空白， 是后向引用，要求再次出现与第1组完全相同的词
backref_pattern = r"\b(\w+)\s+\1\b"

for m in re.finditer(backref_pattern, repeat_text):
    print("重复单词:", m.group(1), " 整体匹配:", repr(m.group(0)))

重复单词: cat  整体匹配: 'cat cat'
重复单词: bird  整体匹配: 'bird bird'


**规则说明**：`\1` 是后向引用，代表第 1 个捕获组实际匹配到的内容，而不是任意单词。因此只有 `cat cat` 和 `bird bird` 这种**同一个词连续出现两次**才会被匹配；`dog cat` 两个词不同，不匹配。

### 3. 日期从“年-月-日”转换为“月-日-年”

In [8]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前:\n" + dates)

# 三个捕获组分别取年、月、日；分隔符可以是 - / . 中的任意一种
date_pattern = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
converted = re.sub(date_pattern, r"\2-\3-\1", dates)
print("\n转换后:\n" + converted)
print("\n行数是否保持不变:", converted.count("\n") == dates.count("\n"))

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后:
05-01-2024
05-02-2024
05-03-2024

行数是否保持不变: True


**规则说明**：`re.sub` 配合捕获组重排顺序——替换串 `\2-\3-\1` 中，`\1` 是年、`\2` 是月、`\3` 是日，于是把“年-月-日”改写为“月-日-年”，三种分隔符都能处理，行数保持不变。

### 4. 电话号码统一为 (123) 456-7890 形式

In [9]:
phones = "联系我：123-456-7890 或 987.654.3210"
phone_pattern = r"(\d{3})[-.](\d{3})[-.](\d{4})"

normalized = re.sub(phone_pattern, r"(\1) \2-\3", phones)
print("统一前:", phones)
print("统一后:", normalized)

统一前: 联系我：123-456-7890 或 987.654.3210
统一后: 联系我：(123) 456-7890 或 (987) 654-3210


**规则说明**：用三个捕获组分别抓 3 位、3 位、4 位号码，分隔符允许 `-` 或 `.`，再用替换串 `(\1) \2-\3` 统一成 `(区号) 前三-后四` 的格式。

---
## 实验3：使用 BeautifulSoup 解析 HTML

使用给定字符串，不依赖网页下载。

In [10]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式
<p class="course">第七讲：结构化数据
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print(type(soup))

<class 'bs4.BeautifulSoup'>


### 1. 页面标题与 h1

In [11]:
print("标题标签文本:", soup.title.string)

h1 = soup.h1
print("h1 的标签名:", h1.name)
print("h1 的文本:", h1.string)

标题标签文本: 实验课程
h1 的标签名: h1
h1 的文本: 数据分析


### 2. 用 find_all 找段落和链接

In [12]:
paragraphs = soup.find_all("p", class_="course")
print("class=course 的段落数量:", len(paragraphs))
for i, p in enumerate(paragraphs, 1):
    print(f"  段落{i}文本:", repr(p.get_text(strip=True)))

links = soup.find_all("a")
print("链接数量:", len(links))
for a in links:
    print("  链接文字:", a.string, " href 属性:", a["href"])

class=course 的段落数量: 2
  段落1文本: '第六讲：正则表达式第七讲：结构化数据讲义六讲义七'
  段落2文本: '第七讲：结构化数据讲义六讲义七'
链接数量: 2
  链接文字: 讲义六  href 属性: https://example.org/lecture06
  链接文字: 讲义七  href 属性: https://example.org/lecture07


**解析说明**：`find_all` 会递归查找所有符合条件的标签。`class_` 参数按 CSS 类筛选段落，`a["href"]` 取标签的属性值。注意原 HTML 的两个 `<p>` 没有显式闭合，解析器会自动处理嵌套关系，但不影响 `find_all` 把它们全部找出。

### 3. 用 get_text 提取可见文本并比较

In [13]:
print("=== 整个文档 get_text() ===")
print(soup.get_text())

print("=== 单个段落 get_text() ===")
print(repr(paragraphs[0].get_text(strip=True)))

print("=== 单个链接 get_text() ===")
print(repr(links[0].get_text()))

=== 整个文档 get_text() ===

实验课程
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七


=== 单个段落 get_text() ===
'第六讲：正则表达式第七讲：结构化数据讲义六讲义七'
=== 单个链接 get_text() ===
'讲义六'


**结果比较**：

- 对整个文档调用 `get_text()` 会拼接所有标签里的可见文本，去掉全部 HTML 标签，得到页面的完整文字内容。
- 对单个段落调用只返回该段落自身的文字。
- 对单个链接调用只返回锚点文字（如“讲义六”），不含 `href` 等属性。

`strip=True` 可以去掉文字前后的空白与换行，便于查看。